In [4]:
from pathlib import Path
from typing import Dict

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

from neuralhydrology.datasetzoo import get_dataset, camelsus
from neuralhydrology.datautils.utils import load_scaler
from neuralhydrology.modelzoo.cudalstm import CudaLSTM
from neuralhydrology.modelzoo.shm import SHM
from neuralhydrology.nh_run import start_run
from neuralhydrology.utils.config import Config
from neuralhydrology.training.basetrainer import BaseTrainer

### Train the LSTM
To start, let's train an lstm for a single basin. If you're curious this is for the Narraguagus River with flow as measured at Cherryfield, Maine. I chose this for consistency with `examples/05-Inspecting-LSTMs`

In [5]:
config_file = Path("1_basin.yml")
# by default we assume that you have at least one CUDA-capable NVIDIA GPU or MacOS with Metal support
# if torch.cuda.is_available() or torch.backends.mps.is_available():
#     start_run(config_file=config_file)

# # fall back to CPU-only mode
# else:
#     start_run(config_file=config_file, gpu=-1)

### Load up trained model
The results of training (model weights, metadata, and optimizer-related data) are saved in `runs`. Let's load them up.

In [6]:
run_dir = Path('runs/test_run_1202_165626')  # this value comes from the output of the above command
!ls $run_dir/model_epoch* | tail -n 3

runs/test_run_1202_165626/model_epoch028.pt
runs/test_run_1202_165626/model_epoch029.pt
runs/test_run_1202_165626/model_epoch030.pt


### Load up trained model
Let's create a new instance of the neural network and then load the trained weights into it.

In [7]:
cudalstm_config = Config(config_file)

# create a new model instance with random weights
cuda_lstm = CudaLSTM(cfg=cudalstm_config)

# load the trained weights into the new model. 
model_path = run_dir / 'model_epoch030.pt'
model_weights = torch.load(str(model_path), map_location='cpu')  # load the weights from the file, creating the weight tensors on CPU
cuda_lstm.load_state_dict(model_weights)  # set the new model's weights to the values loaded from file
cuda_lstm

CudaLSTM(
  (embedding_net): InputLayer(
    (statics_embedding): Identity()
    (dynamics_embeddings): ModuleList(
      (0): Identity()
    )
  )
  (lstm): LSTM(5, 20)
  (dropout): Dropout(p=0.4, inplace=False)
  (head): Regression(
    (net): Sequential(
      (0): Linear(in_features=20, out_features=1, bias=True)
    )
  )
)

### Configuring SHM
Now, let's initialize an SHM model.

In [8]:
shm_config_file = Path("shm_config.yml")
shm_config = Config(shm_config_file)
shm = SHM(cfg=shm_config)

### Fetch the data
Lets instantiate a dataloader containing the data we want

In [9]:
trainer = BaseTrainer(cfg = Config(config_file))
trainer.initialize_training()
loader = trainer.loader

2026-02-19 13:57:29,625: Logging to /Users/danielmckenzie/Documents/Active_Research/Ziyu/neuralhydrology-dcfe/examples/09-Data-Assimilation/runs/test_run_1902_135729/output.log initialized.
2026-02-19 13:57:29,625: ### Folder structure created at /Users/danielmckenzie/Documents/Active_Research/Ziyu/neuralhydrology-dcfe/examples/09-Data-Assimilation/runs/test_run_1902_135729
2026-02-19 13:57:29,626: ### Run configurations for test_run
2026-02-19 13:57:29,626: experiment_name: test_run
2026-02-19 13:57:29,627: train_basin_file: 1_basin.txt
2026-02-19 13:57:29,627: validation_basin_file: 1_basin.txt
2026-02-19 13:57:29,627: test_basin_file: 1_basin.txt
2026-02-19 13:57:29,628: train_start_date: 1999-10-01 00:00:00
2026-02-19 13:57:29,628: train_end_date: 2008-09-30 00:00:00
2026-02-19 13:57:29,628: validation_start_date: 1980-10-01 00:00:00
2026-02-19 13:57:29,629: validation_end_date: 1989-09-30 00:00:00
2026-02-19 13:57:29,629: test_start_date: 1989-10-01 00:00:00
2026-02-19 13:57:29,62

In [10]:
data_point = next(iter(loader))

In [11]:
print(data_point.keys())

dict_keys(['x_d', 'x_d_hindcast', 'x_d_forecast', 'y', 'date'])


In [12]:
print(data_point['x_d'].keys())

dict_keys(['prcp(mm/day)', 'srad(W/m2)', 'tmax(C)', 'tmin(C)', 'vp(Pa)'])


In [13]:
print(data_point['x_d']['tmax(C)'].shape)

torch.Size([256, 365, 1])


In [14]:
print(data_point['x_d']['tmax(C)'][0:5,0:2,:])

tensor([[[ 0.9162],
         [ 0.6870]],

        [[-1.8545],
         [-1.6395]],

        [[-0.4610],
         [-1.8345]],

        [[ 1.5573],
         [ 1.6105]],

        [[ 1.1578],
         [ 1.1825]]])


In [15]:
print(data_point['date'])

[['2006-09-02T00:00:00.000000000' '2006-09-03T00:00:00.000000000'
  '2006-09-04T00:00:00.000000000' ... '2007-08-30T00:00:00.000000000'
  '2007-08-31T00:00:00.000000000' '2007-09-01T00:00:00.000000000']
 ['2003-01-25T00:00:00.000000000' '2003-01-26T00:00:00.000000000'
  '2003-01-27T00:00:00.000000000' ... '2004-01-22T00:00:00.000000000'
  '2004-01-23T00:00:00.000000000' '2004-01-24T00:00:00.000000000']
 ['1998-12-23T00:00:00.000000000' '1998-12-24T00:00:00.000000000'
  '1998-12-25T00:00:00.000000000' ... '1999-12-20T00:00:00.000000000'
  '1999-12-21T00:00:00.000000000' '1999-12-22T00:00:00.000000000']
 ...
 ['2001-04-10T00:00:00.000000000' '2001-04-11T00:00:00.000000000'
  '2001-04-12T00:00:00.000000000' ... '2002-04-07T00:00:00.000000000'
  '2002-04-08T00:00:00.000000000' '2002-04-09T00:00:00.000000000']
 ['2003-03-17T00:00:00.000000000' '2003-03-18T00:00:00.000000000'
  '2003-03-19T00:00:00.000000000' ... '2004-03-13T00:00:00.000000000'
  '2004-03-14T00:00:00.000000000' '2004-03-15T0

In [16]:
shm_parameters = {
            "dd": 5.0,
            "f_thr": 10.0,
            "sumax": 25.0,
            "beta": 5.0,
            "perc": 0.5,
            "kf": 3.0,
            "ki": 5.0,
            "kb": 15.0,
        }

In [19]:
ss, sf, su, si, sb = shm.initialize_states(batch_size=1, device = 'cpu')

In [ ]:
## ToDo:
# 1. Fetch decent parameters, or make them up.
# 1.5 initialize the data loader. 
# 2. use shm.timestep in a for loop.
# 3. How do I connect this to forcings?